# Governor — E0029-QWEN: cross-backend allocation experiment

**Notebook outputs are evidence, not truth.** Every headline number here must be
recomputed from raw artifacts by `scripts/verify_colab_run.py` before it counts.

---

## What this tests

Whether the Governor's allocation mechanism survives a change of reasoning
backend. Everything except the backend is held fixed: benchmark family,
allocation formulation, feature boundary, target, grouped cross-validation,
calibration/evaluation discipline, resource accounting.

## What it does NOT test

- It is **not** a replication of the `gpt-oss-120b` preregistration. That run was
  stopped because a token cap was manufacturing failures, and changing backend
  changes the capability regime.
- It does **not** test maximum attainable accuracy. A weaker model is fine; what
  matters is that some problems are solvable and some are not, so there is
  something to allocate between.
- A positive result here would **not** establish that "Governor works". The
  strongest permitted claim is tied to this benchmark, model, resource axis,
  split, estimator, and awaits its own replication.

## Current scientific status

| | |
|---|---|
| observable ceiling | +0.0588 |
| oracle marginal ranking | +0.0505 [+0.0203, +0.0816] |
| learned ranker (E0027) | +0.0055 [−0.0104, +0.0218] |
| learned ranker (E0028) | +0.0070 [−0.0128, +0.0287] |
| real-LLM advantage | **NOT VERIFIED** |

The allocation formulation is sound — an oracle ranker converts it into a gain
with a CI excluding zero. The learned ranker does not. E0028 diagnosed this as
data starvation, and this experiment asks whether the same pattern appears on a
different backend.

## Why this experiment exists

Two runs have already been abandoned for configuration faults that a short check
would have caught:

- `gpt-oss-120b` at a 2500-token cap: 34% of samples truncated, **42% of those
  produced no code at all**, against 0% of uncapped ones. Because harder problems
  reason longer, the artificial failure rate would have risen with difficulty —
  a confound pointed straight at the quantity being measured.
- Qwen3 with default thinking enabled: 3/3 sampled problems consumed the entire
  budget without emitting code, projecting 191 hours.

Every gate below exists because of a specific failure like these.

## Requirements

| | |
|---|---|
| runtime | preflight ≈ 10 min · pilot ≈ 25 min · full run measured before it starts |
| GPU | **L4 recommended** (Runtime → Change runtime type → L4). T4 works; A100 is overkill for a 1.7B model. If `REQUIRE_GPU` is set and none is present the notebook **fails** rather than falling back. |
| storage | ≈ 2 GB (model weights + artifacts) |
| internet | required — GitHub and Hugging Face |
| Drive | **optional**; local first, Drive only as an archive |

## How to reproduce

```
git checkout <commit printed in section 03>
python scripts/verify_colab_run.py --handoff claude_handoff/
```


## 00 — Experiment identity

Frozen here, read by every later section. No cell hard-codes these.

In [ ]:
EXPERIMENT_ID = "E0029-QWEN"
REPO_URL     = "https://github.com/SYT20/Governor.git"
REPO_REF     = ""          # commit or tag; "" tracks the remote default branch
REQUIRE_GPU  = True        # fails on ABSENT HARDWARE, not on a guess about the client
# PERSISTENCE. The previous run finished, archived nothing, and was lost with
# the VM: 4750 samples and hours of L4 time. Drive is no longer an optional
# archive that runs at the END -- a verified durable sink is a PRECONDITION of
# the full run, and rows are mirrored to it at every batch boundary.
USE_DRIVE       = True     # mount Drive and mirror generated rows to it
ALLOW_EPHEMERAL = False    # True = run with NO durable copy. Rows die with the VM.
RUN_MODE     = "AUTO"      # AUTO | HOSTED_COLAB | VSCODE_COLAB | LOCAL
PILOT_PROBLEMS, PILOT_SAMPLES = 20, 5

# Validate here, once. A wrong type in one of these surfaces four cells later as
# something unrelated -- a REQUIRE_GPU of "True" (a string) is truthy and would
# silently pass a check meant to stop a CPU run.
assert isinstance(EXPERIMENT_ID, str) and EXPERIMENT_ID
assert isinstance(REPO_URL, str) and REPO_URL.startswith(("http", "/"))
assert isinstance(REPO_REF, str)
assert isinstance(REQUIRE_GPU, bool), "REQUIRE_GPU must be a bool, not a string"
assert isinstance(USE_DRIVE, bool), "USE_DRIVE must be a bool, not a string"
assert isinstance(ALLOW_EPHEMERAL, bool), "ALLOW_EPHEMERAL must be a bool"
assert RUN_MODE in ("AUTO", "HOSTED_COLAB", "VSCODE_COLAB", "LOCAL")
assert isinstance(PILOT_PROBLEMS, int) and PILOT_PROBLEMS > 0
assert isinstance(PILOT_SAMPLES, int) and PILOT_SAMPLES > 0

print(f"EXPERIMENT_ID   {EXPERIMENT_ID}")
print(f"REPO_REF        {REPO_REF or '(remote default branch)'}")
print(f"REQUIRE_GPU     {REQUIRE_GPU}")
print(f"USE_DRIVE       {USE_DRIVE}   (archive only — never required)")
print(f"RUN_MODE        {RUN_MODE}")
print(f"PILOT           {PILOT_PROBLEMS} problems x {PILOT_SAMPLES} samples")
print("\nCONFIG_STATUS = PASS")


## 01–03 — One-cell bootstrap: environment, dependencies, repository

Paste-able into a brand-new Colab. Assumes no cwd, no imports, no pip state, no Drive, no PYTHONPATH. Ends with `COLAB_BOOTSTRAP = PASS` or a clear error.

In [ ]:
import os, subprocess, sys, pathlib

WORK = pathlib.Path("/content") if pathlib.Path("/content").is_dir() else pathlib.Path.cwd()
REPO = WORK / "Governor"

def _sh(args, **kw):
    return subprocess.run(args, capture_output=True, text=True, **kw)

# THE UPDATE HAPPENS HERE, IN THE CELL, NOT IN THE SCRIPT IT CALLS.
#
# This cell is the only component guaranteed current: it comes from the notebook
# you opened, while scripts/ comes from whatever the runtime last cloned. An
# earlier colab_bootstrap.py fetched without moving the working tree, so a
# checkout could sit at an old commit indefinitely -- and the fix for that bug
# lives in the very file the stale checkout would not update. Delegating the
# update to the checkout could never break that cycle. Doing it here does.
if not (REPO / ".git").is_dir():
    print(f"cloning {REPO_URL}")
    _sh(["git", "clone", "--quiet", REPO_URL, str(REPO)])
    before = ""
else:
    before = _sh(["git", "rev-parse", "--short", "HEAD"], cwd=REPO).stdout.strip()
    _sh(["git", "fetch", "--all", "--tags", "--force", "--quiet"], cwd=REPO)

if REPO_REF:
    _sh(["git", "checkout", "--quiet", "--force", REPO_REF], cwd=REPO)
else:
    head = _sh(["git", "symbolic-ref", "--quiet", "--short",
                "refs/remotes/origin/HEAD"], cwd=REPO).stdout.strip()
    _sh(["git", "reset", "--hard", "--quiet", head or "origin/main"], cwd=REPO)

after = _sh(["git", "rev-parse", "--short", "HEAD"], cwd=REPO).stdout.strip()
subj = _sh(["git", "log", "-1", "--format=%s"], cwd=REPO).stdout.strip()
if before and before != after:
    print(f"updated  {before} -> {after}   {subj[:60]}")
elif before:
    print(f"current  {after}   {subj[:60]}")
else:
    print(f"cloned   {after}   {subj[:60]}")

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# Prove the tree really moved: these files postdate the bug above, so their
# absence means the update silently failed and nothing downstream will work.
_need = ["scripts/colab_stream.py", "scripts/colab_bootstrap.py",
         "scripts/colab_preflight.py", "scripts/e0029_colab_generate.py"]
_missing = [f for f in _need if not (REPO / f).exists()]
if _missing:
    raise SystemExit(
        f"update did not take — still missing: {_missing}\n"
        f"  HEAD is {after}. Run this once in a cell, then re-run:\n"
        f"      !cd {REPO} && git fetch --all --force && git reset --hard origin/main")
print(f"required files present: {len(_need)}/{len(_need)}")

rc = subprocess.run([sys.executable, "-u", "scripts/colab_bootstrap.py",
                     "--repo", REPO_URL, "--ref", REPO_REF, "--dest", str(REPO)],
                    cwd=REPO, capture_output=True, text=True)
print(rc.stdout)
if rc.stderr.strip():
    print("--- stderr ---"); print(rc.stderr[-2000:])
if rc.returncode != 0:
    raise SystemExit("bootstrap failed — see the output above")


## 03b — Make sure the checkout actually has the E0029 grading code

Two independent routes, checked in this order:

1. **The clone already has it.** If `scripts/durable_sink.py` is present, the
   bootstrap pulled a current `main` and there is nothing to do.
2. **The Drive overlay.** `governor_e0029_overlay.py` in your Drive carries the
   eight files as one sha256-verified blob and writes them into the checkout.

The overlay exists because these files reached Drive before they reached GitHub.
It is idempotent — applying it over an already-current checkout rewrites the
same bytes and re-checks every digest.

If neither route works, **stop here**. The next cell imports `durable_sink`, and
without it nothing guards the expensive run.


In [ ]:
import sys, pathlib, hashlib, shutil, importlib
_root = pathlib.Path.cwd()
_root = _root if (_root / "governor").is_dir() else _root / "Governor"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

NEEDED = ["scripts/durable_sink.py", "scripts/e0029_grade.py",
          "scripts/e0029_analyse.py", "governor/execfeedback/privatetests.py"]
missing = [f for f in NEEDED if not (_root / f).exists()]

if not missing:
    print("checkout already has the E0029 grading code:")
    for f in NEEDED:
        h = hashlib.sha256((_root / f).read_bytes()).hexdigest()[:12]
        print(f"    {f:44s} {h}")
else:
    print(f"missing {len(missing)} file(s) — applying the Drive overlay\n")

    # Mount only if needed. Check the mountpoint trap FIRST: once /content/drive
    # exists as an ordinary directory, drive.mount() fails with "Mountpoint must
    # not already contain files" and the message names a mountpoint, not a cause.
    mnt = pathlib.Path("/content/drive")
    if not (mnt / "MyDrive").is_dir():
        if mnt.exists():
            n = len(list(mnt.iterdir()))
            print(f"/content/drive exists as a plain directory ({n} entries) and "
                  f"is NOT a mount.\nRemoving it so the mount can succeed.")
            shutil.rmtree(mnt)
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:                              # noqa: BLE001
            raise SystemExit(
                f"could not mount Drive: {type(e).__name__}: {e}\n\n"
                "  The overlay lives in Drive, so it cannot be fetched without a\n"
                "  mount. Alternative: the same files are on GitHub main — re-run\n"
                "  the BOOTSTRAP cell to pull a current checkout.") from None

    src = None
    for cand in (pathlib.Path("/content/drive/MyDrive/governor_e0029_overlay.py"),
                 pathlib.Path("/content/drive/MyDrive/governor_e0029/"
                              "governor_e0029_overlay.py")):
        if cand.exists():
            src = cand
            break
    if src is None:
        raise SystemExit(
            "governor_e0029_overlay.py is not in your Drive.\n"
            "  Looked in MyDrive/ and MyDrive/governor_e0029/.\n"
            "  Alternative: re-run the BOOTSTRAP cell to pull it from GitHub.")

    print(f"overlay: {src}  ({src.stat().st_size/1024:.0f} KB)\n")
    sys.path.insert(0, str(src.parent))
    spec = importlib.util.spec_from_file_location("_gov_overlay", src)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    rc = mod.apply(str(_root))
    if rc != 0:
        raise SystemExit(f"overlay failed (exit {rc}) — do not continue")

# Prove it imports, rather than trusting that a file appeared.
for _m in ("scripts.durable_sink", "governor.execfeedback.privatetests"):
    importlib.invalidate_caches()
    importlib.import_module(_m)
print("\nimports verified — durable_sink and privatetests are loadable")


## 00b — Runtime, GPU and Drive: what this environment actually is

The notebook must behave identically under hosted Colab and under the VS Code
Colab extension. Those differ in ways that break naive checks: `google.colab` is
often absent under VS Code **even when the compute is a Colab VM**, and Drive's
OAuth prompt needs an interactive channel VS Code may not provide.

Detection therefore weighs several independent signals rather than reading one
variable — inferring from a single check is how a notebook comes to announce
"not running in Colab" on a machine that plainly is.

Three things are decided separately here, because they are separate facts:

- **runtime kind** — which client is driving the kernel
- **GPU** — actual hardware. `REQUIRE_GPU` fails on an absent card, never on the
  runtime being unrecognised; a Colab VM driven from VS Code has the same card.
- **Drive** — a *precondition* of the full run, verified by round-trip before
  anything expensive starts (cell 00c), and mirrored to at every batch
  boundary. It used to be "an archive, never a dependency"; that rule let a
  completed run report PASS while persisting nothing, and 4750 samples died
  with the VM. Running without a durable copy now requires `ALLOW_EPHEMERAL`.
  not an experiment failure.


In [ ]:
import sys, pathlib
_root = pathlib.Path.cwd()
_root = _root if (_root / "governor").is_dir() else _root / "Governor"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
try:
    from scripts.colab_runtime import (detect_notebook_runtime, detect_drive,
                                       detect_gpu)
except ImportError as e:
    raise SystemExit(
        f"cannot import scripts.colab_runtime ({e}).\n"
        f"  repo looked for at: {_root}  exists={_root.is_dir()}\n"
        f"  Run the BOOTSTRAP cell first — it clones and updates the checkout."
    ) from None

RUNTIME = detect_notebook_runtime()
print(RUNTIME.render())

if RUN_MODE != "AUTO" and RUN_MODE != RUNTIME.kind:
    print(f"\nnote: RUN_MODE={RUN_MODE} overrides detected {RUNTIME.kind}")
    RUNTIME.kind = RUN_MODE

GPU = detect_gpu()
print(f"\nGPU_REQUIRED    {REQUIRE_GPU}")
print(f"GPU_AVAILABLE   {GPU.available}")
print(f"GPU_MODEL       {GPU.name or '-'}  {GPU.memory_gb or ''} {GPU.backend}")
print(f"GPU_STATUS      {'PASS' if (GPU.available or not REQUIRE_GPU) else 'FAIL'}")
if REQUIRE_GPU and not GPU.available:
    raise SystemExit(
        f"GPU_STATUS = FAIL — {GPU.reason}\n"
        f"  REQUIRE_GPU is set, so this stops rather than silently taking ~20x\n"
        f"  longer on CPU. Attach a GPU runtime, or set REQUIRE_GPU = False.")

DRIVE = detect_drive(RUNTIME)
print(f"\nDRIVE_AVAILABLE {DRIVE.available}")
print(f"DRIVE_MOUNTED   {DRIVE.mounted}")
print(f"DRIVE_INTERACTIVE_MOUNT {DRIVE.interactive_mount_supported}")
print(f"  {DRIVE.reason}")
print("\nDrive is an ARCHIVE. The experiment runs on local storage regardless.")
print("\nRUNTIME_STATUS = PASS")


## 00c — Somewhere durable to write, verified before anything expensive

**This is the cell that would have saved the last run.** It mounts Drive, writes
a probe, reads it back, and compares. A directory that merely *exists* proves
nothing — a half-detached mount presents one and swallows every write.

If this cell cannot find a durable sink, the full-run cell will refuse to start.
That is deliberate. A run that completes and then evaporates is not a cheaper
success than a run that never started; it is the same outcome, minus the GPU
hours.


In [ ]:
import sys, pathlib, os
_root = pathlib.Path.cwd()
_root = _root if (_root / "governor").is_dir() else _root / "Governor"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from scripts.durable_sink import find_durable_sink, verify_writable
from scripts.colab_runtime import detect_notebook_runtime

RT = detect_notebook_runtime()
print(f"runtime: {RT.kind}\n")

# Mount FIRST, while there is still nothing to lose. Only hosted Colab can raise
# the OAuth prompt; under the VS Code extension it usually has nowhere to appear,
# which is exactly how the previous mount failed.
if USE_DRIVE and not pathlib.Path("/content/drive/MyDrive").is_dir():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:                                  # noqa: BLE001
        print(f"could not mount Drive: {type(e).__name__}: {e}\n")

SINK, tried = find_durable_sink()
for st in tried:
    print(f"  [{'OK  ' if st.ok else 'FAIL'}] {st.path}")
    print(f"         {st.kind} -- {st.reason}")

if SINK is not None:
    print(f"\nDURABLE SINK: {SINK.path}  ({SINK.round_trip_ms} ms round-trip)")
    print("Generated rows are mirrored here at every batch boundary.")
    print("If this VM dies mid-run, re-running resumes from the durable copy.")
elif ALLOW_EPHEMERAL:
    print("\nNO DURABLE SINK — proceeding because ALLOW_EPHEMERAL = True.")
    print("Everything this run produces dies with the VM. Download results/")
    print("the moment generation finishes.")
else:
    raise SystemExit(
        "\nNO DURABLE SINK, and ALLOW_EPHEMERAL is False.\n\n"
        "  The full run will refuse to start, on purpose.\n\n"
        "  If you are in the VS Code Colab extension, the Drive OAuth prompt has\n"
        "  nowhere to appear. Open this notebook in HOSTED Colab instead\n"
        "  (colab.research.google.com, in a browser) and mount Drive there.\n\n"
        "  Or point GOVERNOR_SINK at any persistent directory:\n"
        "      import os; os.environ['GOVERNOR_SINK'] = '/your/persistent/dir'\n\n"
        "  Or set ALLOW_EPHEMERAL = True in cell 2 to accept losing the run.")


## 00d — VS Code / Colab compatibility

Everything the pipeline relies on, checked in one place: plain execution,
subprocess, git, filesystem, the text loader, repository imports, and GPU. Under
the VS Code extension each of these can behave differently from hosted Colab, and
a failure here is far cheaper to read than the same failure surfacing inside a
four-hour generation run.


In [ ]:
import sys, subprocess, pathlib, tempfile, json

checks, failures = [], []

def check(name, fn):
    try:
        detail = fn()
        checks.append((name, True, str(detail)[:60]))
    except Exception as e:                                  # noqa: BLE001
        checks.append((name, False, f"{type(e).__name__}: {e}"))
        failures.append(name)

check("python execution", lambda: sys.version.split()[0])
check("subprocess", lambda: subprocess.run(
    [sys.executable, "-c", "print('ok')"], capture_output=True, text=True,
    check=True).stdout.strip())
check("git", lambda: subprocess.run(
    ["git", "--version"], capture_output=True, text=True, check=True).stdout.strip())

def _fs():
    with tempfile.TemporaryDirectory() as td:
        p = pathlib.Path(td) / "probe"
        p.write_text("x")
        assert p.read_text() == "x"
    return "read/write/delete ok"
check("filesystem", _fs)

check("repo present", lambda: f"{RUNTIME.repo} {pathlib.Path(RUNTIME.repo).is_dir()}")
check("text loader", lambda: __import__(
    "scripts.colab_text_loader", fromlist=["load_package_module"]).__name__)
check("import parity", lambda: __import__(
    "scripts.colab_text_loader", fromlist=["verify_import_parity"]
    ).verify_import_parity("governor.gate.m2_interface",
                           pathlib.Path(RUNTIME.repo))["ok"])
check("repository import", lambda: __import__(
    "governor.harness.traps", fromlist=["run_trap_checks"]).__name__)
check("streamer", lambda: __import__(
    "scripts.colab_stream", fromlist=["run_streamed"]).__name__)
check("gpu", lambda: f"{GPU.available} {GPU.name or '-'}")

for name, ok, detail in checks:
    print(f"  [{'  ok  ' if ok else ' FAIL '}] {name:<20} {detail}")

status = "PASS" if not failures else "FAIL"
print(f"\nVSCODE_COLAB_COMPATIBILITY = {status}")
if failures:
    print(f"  failing: {failures}")
    print(f"  runtime {RUNTIME.kind}  cwd {RUNTIME.cwd}  commit {RUNTIME.commit}")
    raise SystemExit(f"compatibility check failed: {failures}")


## 04–05 — Text-based source loader and import parity

Loads modules from source **text**, independent of notebook state, then loads the same modules by ordinary import and requires the two to agree. A divergence means the notebook is reading a stale cache rather than the repository.

In [ ]:
from scripts.colab_text_loader import (load_package_module, verify_import_parity,
                                       repo_root, file_sha256)

TARGETS = ["governor.harness.ledger", "governor.harness.traps",
           "governor.harness.drivers", "governor.gate.m2_interface",
           "governor.execution.executor", "governor.phase4.statemgr",
           "governor.execfeedback.sandbox", "governor.execfeedback.preflight",
           "governor.execfeedback.publictests"]

bad = []
for t in TARGETS:
    r = verify_import_parity(t, repo_root())
    print(f"  {'ok  ' if r['ok'] else 'FAIL'}  {t:<40} {r['names']:>3} names  sha {r['source_sha256'][:8]}")
    if not r["ok"]:
        bad.append((t, r["problems"]))

assert not bad, f"import parity failed: {bad}"
print("\n  all modules load identically via text loader and normal import")

## 06–10 — Every preflight gate

Environment, loader parity, restart safety in a fresh subprocess, sandbox containment, information boundary, M2 contract, checkpoint resume, frozen split, and the project's own test suite. Ends with `READY FOR FULL RUN` or names the first failing gate.

In [ ]:
import sys, pathlib, subprocess

# Context FIRST, unconditionally. Whatever happens next, the output carries
# enough to diagnose it: where we are, which commit, and whether the helper
# this cell needs is actually present. Three exchanges were spent asking for
# an error message that the notebook could simply have printed itself.
_cwd = pathlib.Path.cwd()
_repo = _cwd if (_cwd / "governor").is_dir() else _cwd / "Governor"
_helper = _repo / "scripts" / "colab_stream.py"
def _git_head(path):
    """Never raise from a diagnostic. subprocess.run(cwd=...) raises
    FileNotFoundError when the directory is absent -- which is precisely the
    case this line exists to report, so it would crash before printing."""
    if not path.is_dir():
        return "(directory does not exist)"
    try:
        r = subprocess.run(["git", "log", "--oneline", "-1"], cwd=str(path),
                           capture_output=True, text=True)
        return r.stdout.strip() or "(not a git repo)"
    except Exception as e:                            # noqa: BLE001
        return f"(git unavailable: {type(e).__name__})"

_commit = _git_head(_repo)
print(f"cwd     {_cwd}")
print(f"repo    {_repo}   exists={_repo.is_dir()}")
print(f"commit  {_commit}")
print(f"helper  {_helper.name} present={_helper.exists()}")

if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

try:
    from scripts.colab_stream import run_streamed
except ImportError as e:
    # Distinguish the two causes rather than asserting one. Telling someone the
    # clone is stale when they simply have not run the bootstrap yet sends them
    # after the wrong thing.
    if not _repo.is_dir():
        why = ("The repository is not checked out here. Run the BOOTSTRAP cell "
               "first — it clones and sets the working directory.")
    elif not _helper.exists():
        why = ("The checkout predates scripts/colab_stream.py. Re-run the "
               "BOOTSTRAP cell; it resets the tree to the remote default branch, "
               "which versions before 0f33a24 did not do.")
    else:
        why = f"The file exists but will not import: {e}"
    raise SystemExit(f"cannot import run_streamed.\n  {why}") from None

import json

rc = run_streamed([sys.executable, "-u", "scripts/colab_preflight.py",
                   *(["--require-gpu"] if REQUIRE_GPU else []),
                   "--json", "results/colab_preflight.json"],
                  heartbeat_s=30, prefix="  ")

PREFLIGHT_OK = rc == 0
rep = pathlib.Path("results/colab_preflight.json")
if rep.exists():
    gates = json.loads(rep.read_text())["gates"]
    print("\nGATES")
    for g in gates:                       # every gate, not only the failures --
        mark = " ok " if g["ok"] else "FAIL"   # a passing list is also evidence
        print(f"  [{mark}] {g['name']:<26} {g['detail'][:70]}")
else:
    print("\nno results/colab_preflight.json was written — the preflight did not "
          "reach its end. The streamed output above is the only record.")

print(f"\nPREFLIGHT_OK = {PREFLIGHT_OK}")
if not PREFLIGHT_OK:
    raise SystemExit("DO NOT START FULL RUN — see the failing gates above")


## 08 — Backend: load the real model and run real inference

Not an import check. Loads weights, generates twice to prove reuse, and records load time, latency, token counts and peak memory. If the model does not fit it reports `MODEL DOES NOT FIT` and stops — it never silently swaps model.

In [ ]:
import json, time, gc, pathlib, inspect
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# A previous run may have installed a torch that does not match this runtime's
# torchvision. `Restart session` does NOT undo that -- Colab keeps installed
# packages in the VM -- and the failure presents as "Could not import module
# 'Qwen3ForCausalLM'", which names the model rather than the broken vision stack.
import subprocess, sys
_abi = subprocess.run([sys.executable, "-c",
    "import torch, torchvision; torch.ops.torchvision.nms; print('OK')"],
    capture_output=True, text=True)
if "OK" not in _abi.stdout:
    raise SystemExit("torch/torchvision ABI mismatch — re-run the bootstrap cell "
                     "(it repairs this), or Runtime -> Disconnect and DELETE runtime.")

cfg = json.loads(pathlib.Path("configs/colab_model.json").read_text())
print(json.dumps({k: v for k, v in cfg.items() if not k.startswith("_")}, indent=1))

device = "cuda" if torch.cuda.is_available() else "cpu"
if REQUIRE_GPU and device != "cuda":
    raise SystemExit("GPU required but unavailable — failing rather than falling back")

t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(cfg["model_name"], revision=cfg["revision"])
_p = inspect.signature(AutoModelForCausalLM.from_pretrained).parameters
_dk = "dtype" if "dtype" in _p else "torch_dtype"   # renamed in transformers 4.56
model = AutoModelForCausalLM.from_pretrained(
    cfg["model_name"], revision=cfg["revision"],
    **{_dk: getattr(torch, cfg["dtype"])}).to(device)
model.eval()
load_s = time.perf_counter() - t0

# REAL inference, twice. Loading proves nothing about generating, and the second
# call proves the model is reused rather than silently reloaded.
prompt = tok.apply_chat_template(
    [{"role": "user", "content": "Write a Python program that reads an integer n "
      "from stdin and prints the sum 1..n. Respond with ONLY a ```python block."}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)

runs = []
for i in range(2):
    enc = tok(prompt, return_tensors="pt").to(device)
    t1 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=256, do_sample=True,
                             temperature=cfg["generation"]["temperature"],
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    runs.append({"latency_s": time.perf_counter() - t1,
                 "prompt_tokens": int(enc["input_ids"].shape[1]),
                 "completion_tokens": int(out.shape[1] - enc["input_ids"].shape[1])})
text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else None
MODEL_META = {"model_name": cfg["model_name"], "revision": cfg["revision"],
              "dtype": cfg["dtype"], "device": device, "load_seconds": load_s,
              "generations": runs, "peak_gpu_gb": peak,
              "torch": torch.__version__, "cuda": torch.version.cuda}
pathlib.Path("results/colab_model_meta.json").write_text(json.dumps(MODEL_META, indent=1))

print(f"\nloaded in {load_s:.1f}s on {device}")
for i, r in enumerate(runs, 1):
    print(f"  generation {i}: {r['completion_tokens']} tokens in {r['latency_s']:.2f}s")
print(f"  reuse: second call {runs[0]['latency_s']/max(runs[1]['latency_s'],1e-9):.1f}x "
      f"the first (>1 means the model was reused, not reloaded)")
if peak:
    print(f"  peak GPU {peak:.2f} GB")
print(f"  sample output: {text[:70]!r}")

# RELEASE. Every later stage runs the model in a SUBPROCESS with its own copy,
# so holding this one only shrinks the memory available to them. Three resident
# copies plus a KV cache is how a 1.7B model came to OOM a 23.7 GB L4.
del model, tok
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"\nreleased — GPU: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")


## 08b — Sequential vs batched: measured on THIS GPU, not assumed

Batching is normally assumed to win on a GPU. Measured on Apple Silicon via MLX it
**lost** — 0.91x, slightly slower than sequential:

```
sequential    93.3s   1875 tokens   20.1 tok/s
batched      102.6s   2306 tokens   22.5 tok/s     0.91x
```

The token counts show why. A batch runs until its **longest** sequence finishes, so
short completions spend steps generating padding — batched did 1.23x the work for
the same result. With this workload's length distribution (median 212, p95 1128)
that waste is severe.

MLX on unified memory is bandwidth-bound at batch size 1, so there is no idle
compute for batching to fill. A CUDA GPU is the opposite, and the mechanism that
makes batching win there is simply absent on Apple Silicon — so the local result
says nothing about Colab. **This cell measures it here instead of extrapolating.**

It also checks whether batched sampling still yields **independent** samples. If a
shared sampler correlates draws, ten samples per problem stop being independent
and the allocation measurement is corrupted — a correctness failure that would
otherwise look like a speedup.


In [ ]:
import sys, pathlib
# Defensive: the cell may be run before/without the bootstrap, or against a
# checkout that predates this module. Say which, rather than raising a bare
# ModuleNotFoundError that names a file the user has no reason to know about.
_root = pathlib.Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
try:
    from scripts.colab_stream import run_streamed
except ImportError:
    raise SystemExit(
        "scripts/colab_stream.py is missing from this checkout.\n"
        "  The clone is stale. Re-run the BOOTSTRAP cell -- it now resets the\n"
        "  working tree to the remote's default branch, which earlier versions\n"
        "  did not do (they fetched without moving anything)."
    ) from None

# Release GPU memory held by earlier cells before a subprocess loads its own.
import gc
try:
    import torch
    for _n in ("model", "tok", "_m", "qwen"):
        globals().pop(_n, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        _f, _t = torch.cuda.mem_get_info()
        print(f"GPU: {_f/1e9:.1f} GB free of {_t/1e9:.1f} GB")
except ImportError:
    pass

import json

rc = run_streamed([sys.executable, "-u", "scripts/colab_batch_bench.py",
                   "--n", "8", "--cap", "1024"], heartbeat_s=30, prefix="  ")

bp = pathlib.Path("results/batch_bench.json")
if rc != 0 or not bp.exists():
    print(f"\nbenchmark did not complete (exit {rc}). Not fatal: the full run "
          f"batches regardless. Continue to the pilot.")
    GENERATION_MODE, BENCH = "BATCHED", None
else:
    BENCH = json.loads(bp.read_text())
    GENERATION_MODE = BENCH["verdict"]
    print(f"\nGENERATION_MODE (measured here) = {GENERATION_MODE}")
    print(f"projected {BENCH['projected_hours_4750']:.1f} h for 4750 samples")


## 11 — Pilot, and the truncation guard

20 problems × 5 samples. Measures the completion-length distribution and **chooses the generation ceiling from evidence**. This gate exists because a 2500-token cap once produced 42% empty outputs while 0% of uncapped samples did.

In [ ]:
import sys, pathlib
# Defensive: the cell may be run before/without the bootstrap, or against a
# checkout that predates this module. Say which, rather than raising a bare
# ModuleNotFoundError that names a file the user has no reason to know about.
_root = pathlib.Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
try:
    from scripts.colab_stream import run_streamed
except ImportError:
    raise SystemExit(
        "scripts/colab_stream.py is missing from this checkout.\n"
        "  The clone is stale. Re-run the BOOTSTRAP cell -- it now resets the\n"
        "  working tree to the remote's default branch, which earlier versions\n"
        "  did not do (they fetched without moving anything)."
    ) from None

# Release GPU memory held by earlier cells before a subprocess loads its own.
import gc
try:
    import torch
    for _n in ("model", "tok", "_m", "qwen"):
        globals().pop(_n, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        _f, _t = torch.cuda.mem_get_info()
        print(f"GPU: {_f/1e9:.1f} GB free of {_t/1e9:.1f} GB")
except ImportError:
    pass

import json

# Expect within ~1 minute:  loading Qwen... -> todo N of M -> prompt length ...
# then a `batch 1:` line. If only heartbeats appear for several minutes with no
# batch line, it is stuck -- interrupt and report what printed.
rc = run_streamed([sys.executable, "-u", "scripts/e0029_colab_generate.py",
                   "--pilot", str(PILOT_PROBLEMS), str(PILOT_SAMPLES),
                   "--batch-size", "16"], heartbeat_s=20, prefix="  ")
PILOT_OK = rc == 0

rep = pathlib.Path("results/E0029-QWEN-preflight.json")
if rep.exists():
    r = json.loads(rep.read_text())
    print("\nPILOT MEASUREMENTS")
    for k in ("n", "median_completion", "p95_completion", "max_completion",
              "truncation_rate", "empty_rate", "solve_rate", "median_latency"):
        if k in r:
            print(f"  {k:<20} {r[k]}")
    if r.get("problems"):
        print("\nFAILING CHECKS:")
        for p in r["problems"]:
            print(f"  - {p}")

print(f"\nPILOT_OK = {PILOT_OK}")
if not PILOT_OK:
    raise SystemExit("pilot failed — full run refused; the failing checks are above")


## 12–13 — Full experiment with continuous checkpointing

Every sample is appended to JSONL immediately and flushed. A restart detects completed samples and skips them; a torn final line is survivable. No sample is ever written twice.

In [ ]:
import sys, pathlib
# Defensive: the cell may be run before/without the bootstrap, or against a
# checkout that predates this module. Say which, rather than raising a bare
# ModuleNotFoundError that names a file the user has no reason to know about.
_root = pathlib.Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
try:
    from scripts.colab_stream import run_streamed
except ImportError:
    raise SystemExit(
        "scripts/colab_stream.py is missing from this checkout.\n"
        "  The clone is stale. Re-run the BOOTSTRAP cell -- it now resets the\n"
        "  working tree to the remote's default branch, which earlier versions\n"
        "  did not do (they fetched without moving anything)."
    ) from None

# Release GPU memory held by earlier cells before a subprocess loads its own.
import gc
try:
    import torch
    for _n in ("model", "tok", "_m", "qwen"):
        globals().pop(_n, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        _f, _t = torch.cuda.mem_get_info()
        print(f"GPU: {_f/1e9:.1f} GB free of {_t/1e9:.1f} GB")
except ImportError:
    pass

import json

rep = pathlib.Path("results/E0029-QWEN-preflight.json")
assert rep.exists(), "no pilot report — run the pilot cell first"
_r = json.loads(rep.read_text())
assert _r.get("ok"), f"pilot did not pass: {_r.get('problems')}"

# Batch size 16 — the value the pilot validated. The KV cache scales with it and
# is what fills the card, so raising it here would put hours of work on an
# untested memory footprint. Resumable: re-run to skip completed samples.
rc = run_streamed([sys.executable, "-u", "scripts/e0029_colab_generate.py",
                   "--full", "--batch-size", "16"]
                  + (["--allow-ephemeral"] if ALLOW_EPHEMERAL else []),
                  heartbeat_s=60, prefix="  ")
print(f"\nFULL_RUN_EXIT = {rc}")

cache = pathlib.Path("results/e0029_colab_generations.jsonl")
if cache.exists():
    n = sum(1 for _ in open(cache))
    print(f"samples on disk: {n} / {475*10}")
    if rc != 0:
        print("Interrupted. Re-run this cell — completed samples are skipped.")


## 14–16 — Grade against PRIVATE tests, then analyse

The generated samples carry only **public**-test outcomes. Those are the
Governor's *features*. Scoring against them too would mean the experiment marks
its own homework, which is the exact failure the preregistration forbids.

So this runs two stages:

1. **`e0029_grade.py`** executes every sample against LiveCodeBench's *private*
   tests and writes an independent hidden label. CPU-only — no GPU needed.
2. **`e0029_analyse.py`** joins features to that label and runs five gates in
   order: data integrity, feature/label separation, ceiling, predictor,
   Governor. **A failing gate stops the run** rather than reporting a number
   computed past it.

> The earlier version of this cell ran `e0028_marginal_ranker.py`, which
> hardcodes `results/E0028_rich.json`. It re-ran E0028, printed split 207/193
> instead of 250/225, reproduced E0028's numbers exactly, and ignored all 4750
> Qwen rows. Gate 0 now treats 207/193 as a named fatal error.


In [ ]:
import sys, pathlib
_root = pathlib.Path.cwd()
_root = _root if (_root / "governor").is_dir() else _root / "Governor"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
try:
    from scripts.colab_stream import run_streamed
except ImportError:
    raise SystemExit(
        "scripts/colab_stream.py is missing from this checkout.\n"
        "  The clone is stale. Re-run the BOOTSTRAP cell -- it resets the working\n"
        "  tree to the remote's default branch."
    ) from None

raw = pathlib.Path("results/e0029_colab_generations.jsonl")
if not raw.exists():
    raise SystemExit("no generations yet -- run the full experiment cell first")

n = sum(1 for _ in open(raw))
print(f"raw generation rows: {n}\n")

# ---- stage 1: the hidden label -----------------------------------------
# Public tests produced the features. Grading against them would score the
# experiment with its own input. This executes the PRIVATE tests instead.
print("=" * 68)
print("STAGE 1 -- grading against private tests (CPU-only, resumable)")
print("=" * 68)
rc = run_streamed([sys.executable, "-u", "scripts/e0029_grade.py"],
                  heartbeat_s=30, prefix="  ")
if rc != 0:
    raise SystemExit(f"grading failed (exit {rc}); analysis not attempted")

graded = pathlib.Path("results/E0029_QWEN_graded.jsonl")
print(f"\ngraded rows: {sum(1 for _ in open(graded))}")

# ---- stage 2: the gated analysis ---------------------------------------
# Freeze on calibration, commit, then evaluate once. Running both phases in one
# process is what makes `frozen_before_heldout` unfalsifiable, so the freeze is
# a separate invocation that writes an artifact the evaluation must find.
print("\n" + "=" * 68)
print("STAGE 2a -- freeze the operating point on calibration")
print("=" * 68)
rc = run_streamed([sys.executable, "-u", "scripts/e0029_analyse.py", "--freeze"],
                  heartbeat_s=30, prefix="  ")
if rc != 0:
    raise SystemExit(f"a gate stopped the freeze (exit {rc}) -- see above")

print("\n" + "=" * 68)
print("STAGE 2b -- apply it once to the held-out set")
print("=" * 68)
rc = run_streamed([sys.executable, "-u", "scripts/e0029_analyse.py",
                   "--json-out", "results/E0029_result.json"],
                  heartbeat_s=30, prefix="  ")
print(f"\nANALYSIS_EXIT = {rc}")
print("  0 = PASS   1 = a gate stopped the run   2 = ran, verdict not PASS")

print("\nNumbers above are evidence, not truth. verify_colab_run.py re-derives")
print("every one of them from raw artifacts before any of it counts.")


## 16b — Get the data off this VM

**The generated rows are the expensive artifact.** They took GPU-hours; the
grading and analysis downstream take minutes and can be re-run any time. A VM
that disappears with the only copy of them costs the whole run — which is what
happened once already: `DRIVE_ARCHIVE: SKIPPED`, the mount raised `ValueError`
under the VS Code extension, and the 4750 rows stayed on a machine that was
later recycled.

So this cell writes one self-contained bundle and tells you, for *this* runtime,
how to actually retrieve it. It never assumes Drive.


In [ ]:
import sys, pathlib, json, hashlib, tarfile, shutil
_root = pathlib.Path.cwd()
_root = _root if (_root / "governor").is_dir() else _root / "Governor"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from scripts.colab_runtime import detect_notebook_runtime, detect_drive

WANT = ["results/e0029_colab_generations.jsonl",
        "results/E0029_QWEN_graded.jsonl",
        "results/E0029_result.json",
        "results/E0029_feature_audit.json",
        "results/E0029_frozen.json",
        "results/e0029_problems.json"]

present = [p for p in WANT if (_root / p).exists()]
missing = [p for p in WANT if p not in present]
if not present:
    raise SystemExit("nothing to recover yet -- run the generation cell first")

out = _root / "claude_handoff"
out.mkdir(exist_ok=True)
bundle = out / "E0029_recovery.tar.gz"

manifest = {}
for p in present:
    b = (_root / p).read_bytes()
    manifest[p] = {"bytes": len(b), "sha256": hashlib.sha256(b).hexdigest(),
                   "rows": sum(1 for _ in open(_root / p)) if p.endswith(".jsonl") else None}
(out / "E0029_manifest.json").write_text(json.dumps(manifest, indent=2))

with tarfile.open(bundle, "w:gz") as tf:
    for p in present:
        tf.add(_root / p, arcname=p)
    tf.add(out / "E0029_manifest.json", arcname="E0029_manifest.json")

size_mb = bundle.stat().st_size / 1e6
print("bundled:")
for p, m in manifest.items():
    r = f"{m['rows']} rows" if m["rows"] is not None else f"{m['bytes']:,} B"
    print(f"    {p:44s} {r:>14s}  {m['sha256'][:12]}")
if missing:
    print("\nnot present (fine if that stage has not run):")
    for p in missing:
        print(f"    {p}")
print(f"\nbundle: {bundle}  ({size_mb:.1f} MB)")
print(f"sha256: {hashlib.sha256(bundle.read_bytes()).hexdigest()}")

# ---- retrieval, by what this runtime can actually do --------------------
rt = detect_notebook_runtime()
dr = detect_drive(rt)
print(f"\nruntime: {rt.kind}")

done = False
if dr.mounted and dr.available:
    dest = pathlib.Path(dr.path) / "governor_e0029"
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(bundle, dest / bundle.name)
    print(f"  COPIED TO DRIVE: {dest / bundle.name}")
    done = True
elif dr.interactive_mount_supported:
    print("  Drive is not mounted but this runtime can mount it. To use it:")
    print("      from google.colab import drive; drive.mount('/content/drive')")
    print("  then re-run this cell.")
else:
    print(f"  Drive unavailable here: {dr.reason}")

if rt.kind == "HOSTED_COLAB":
    try:
        from google.colab import files
        print("\n  Starting browser download...")
        files.download(str(bundle))
        done = True
    except Exception as e:                                  # noqa: BLE001
        print(f"  browser download unavailable ({type(e).__name__})")

if not done:
    print("\n  RETRIEVE IT MANUALLY -- the bundle is a normal file on this VM:")
    print(f"      {bundle}")
    if rt.kind == "VSCODE_COLAB":
        print("    In VS Code: open the Explorer, navigate to that path,")
        print("    right-click the file and choose Download.")
        print("    (This path does NOT need Drive, which is why it is the")
        print("     fallback -- the mount is the part that failed before.)")
    print("\n    Then, on your local machine:")
    print("      tar -xzf E0029_recovery.tar.gz -C /path/to/Governor")
    print("      python scripts/e0029_analyse.py --freeze   # if not already frozen")

print("\nOnce results/e0029_colab_generations.jsonl exists locally, everything")
print("downstream runs on CPU in minutes. The GPU work does not need repeating.")


## 17–19 — Archive to Drive, and the handoff for verification

**Run this cell as often as you like** — during generation, after it, or after an
interrupt. A partial archive is useful; a missing one is not, and the likeliest
end to a multi-hour Colab session is an unplanned one.

**The mount happens here, in the notebook, and that is not incidental.**
`google.colab.drive` is injected into the notebook *kernel*, and mounting needs
the kernel's interactive channel to show the OAuth prompt. A helper script run as
a subprocess has neither, so a mount attempted from there can only fail — and
would fail reporting "not running in Colab" on a machine that plainly is. The
notebook mounts; `colab_archive.py` only *verifies*, by writing a probe file
rather than trusting that the directory exists.

Drive stays optional. Everything is written locally first, and if the mount is
absent or unwritable the archive still completes and reports
`DRIVE_ARCHIVE = UNAVAILABLE`. Claiming persistence that did not happen is worse
than having none, because you find out only once the runtime is gone.


In [ ]:
import sys, pathlib, json, traceback
_root = pathlib.Path.cwd()
_root = _root if (_root / "governor").is_dir() else _root / "Governor"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from scripts.colab_stream import run_streamed
from scripts.colab_runtime import detect_notebook_runtime, detect_drive
from scripts.durable_sink import mount_blocked_by_stray_dir

_rt = RUNTIME if "RUNTIME" in globals() else detect_notebook_runtime()
_dr = detect_drive(_rt)
DRIVE_ARCHIVE = "DISABLED" if not USE_DRIVE else None
mounted = _dr.mounted and _dr.available

# Is the mountpoint already poisoned? Once /content/drive exists as an ordinary
# directory, every mount fails and the error names a mountpoint, not a cause.
_blocked, _why = mount_blocked_by_stray_dir()
if _blocked:
    print("MOUNT IS BLOCKED:\n    " + _why + "\n")

if USE_DRIVE and not mounted:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        _dr = detect_drive(_rt)
        mounted = _dr.mounted and _dr.available
        print(f"drive mount attempted: mounted={mounted}")
    except Exception as e:                                  # noqa: BLE001
        # PRINT THE MESSAGE. The previous version printed only the exception
        # TYPE and discarded str(e); the run that lost 4750 rows logged
        # "drive mount unavailable (ValueError)" and nothing else, so the cause
        # is permanently unrecoverable. For drive.mount the message IS the
        # diagnosis -- e.g. "Mountpoint must not already contain files".
        print(f"DRIVE MOUNT FAILED: {type(e).__name__}: {e}\n")
        traceback.print_exc()

rc = run_streamed([sys.executable, "-u", "scripts/colab_archive.py"]
                  + (["--drive"] if mounted else []),
                  heartbeat_s=30, prefix="  ")

hp = pathlib.Path("claude_handoff")
if not hp.exists():
    DRIVE_ARCHIVE = "UNAVAILABLE"
    print("\nARCHIVE_STATUS = NO DATA YET — run the experiment first")
else:
    s = json.loads((hp / "experiment_summary.json").read_text())
    dstat = json.loads((hp / "drive_status.json").read_text())
    DRIVE_ARCHIVE = DRIVE_ARCHIVE or dstat["status"]
    print(f"\nSTATUS          {s['status']}")
    print(f"rows            {s['rows']}")
    print(f"problems        {s['problems_seen']}/{s.get('problems_expected')}")
    print(f"DRIVE_ARCHIVE   {DRIVE_ARCHIVE}   {dstat.get('path', dstat.get('reason',''))}")
    print(f"HANDOFF         {hp.resolve()}")

    # A durable copy exists, or it does not. Nothing else decides this.
    # The previous version printed "ARCHIVE_STATUS = PASS (Drive status is
    # separate and not required)" over an archive holding nothing, which is
    # what made destroying the VM look like a safe action.
    durable = (DRIVE_ARCHIVE == "OK")
    if durable:
        print("\nARCHIVE_STATUS = PASS — a durable copy exists off this VM")
    elif globals().get("ALLOW_EPHEMERAL"):
        print("\nARCHIVE_STATUS = EPHEMERAL (accepted via ALLOW_EPHEMERAL)")
        print("  Nothing is stored off this VM. Download results/ NOW.")
    else:
        print("\n" + "=" * 68)
        print("ARCHIVE_STATUS = AT RISK — NOTHING IS STORED OFF THIS VM")
        print("=" * 68)
        print(f"  {s['rows']} generated rows exist ONLY on this machine.")
        print("  They are the expensive artifact; grading and analysis are")
        print("  minutes of CPU and can be redone from them at any time.")
        print("\n  DO NOT CLOSE OR DELETE THIS RUNTIME until you have a copy.")
        print("\n  Fastest route — run the recovery cell above (16b), or here:")
        print("      from google.colab import files")
        print("      files.download('claude_handoff/E0029_recovery.tar.gz')")
        if _blocked:
            print("\n  Drive is blocked by a stray directory. To fix and retry:")
            print("      import shutil; shutil.rmtree('/content/drive')")
            print("      from google.colab import drive; drive.mount('/content/drive')")
            print("      # then re-run this cell")

    print("\nVerify independently — the notebook's numbers are evidence, not truth:")
    print("  python scripts/verify_colab_run.py --handoff claude_handoff/")
